In [1]:
import os, json, random, torch, torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report

ROOT = r"C:\Users\ok\prism-dr\data\multi_cancer"
N, EP = 150, 3

os.makedirs("models", exist_ok=True)

while True:
    d = [x for x in os.listdir(ROOT) if os.path.isdir(os.path.join(ROOT, x))]
    if len(d) == 1:
        ROOT = os.path.join(ROOT, d[0])
    else:
        break
classes = sorted(d)
print("Classes found:", classes)

items = []
for ci, c in enumerate(classes):
    fs = [os.path.join(r, f) for r, _, fl in os.walk(os.path.join(ROOT, c))
          for f in fl if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    random.seed(0)
    random.shuffle(fs)
    items += [(f, ci) for f in fs[:N]]

random.shuffle(items)
n = int(0.8 * len(items))
tr_i, te_i = items[:n], items[n:]

norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
T_tr = transforms.Compose([transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(),
                           transforms.RandomRotation(15), transforms.ToTensor(), norm])
T_te = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), norm])

class DS(Dataset):
    def __init__(s, it, t): s.it, s.t = it, t
    def __len__(s): return len(s.it)
    def __getitem__(s, i):
        f, y = s.it[i]
        return s.t(Image.open(f).convert("RGB")), y

dl = DataLoader(DS(tr_i, T_tr), batch_size=32, shuffle=True)
dt = DataLoader(DS(te_i, T_te), batch_size=32)

m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for p in m.parameters(): p.requires_grad = False
for p in m.layer4.parameters(): p.requires_grad = True
m.fc = nn.Linear(512, len(classes))
opt = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], 1e-3)
lf = nn.CrossEntropyLoss()

print("Training started...")
for e in range(EP):
    m.train()
    tot = 0
    for x, y in dl:
        opt.zero_grad()
        l = lf(m(x), y)
        l.backward()
        opt.step()
        tot += l.item()
    print(f"Epoch {e+1} loss: {tot/len(dl):.3f}")

m.eval()
P, Y = [], []
with torch.no_grad():
    for x, y in dt:
        P += m(x).argmax(1).tolist()
        Y += y.tolist()

print("\n--- Results ---")
print(classification_report(Y, P, target_names=classes))

torch.save(m.state_dict(), "models/m2_resnet18.pt")
json.dump(classes, open("models/m2_classes.json", "w"))
print("DONE - Model saved!")

Classes found: ['ALL', 'Brain Cancer', 'Breast Cancer', 'Cervical Cancer', 'Kidney Cancer', 'Lung and Colon Cancer', 'Lymphoma', 'Oral Cancer']
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\ok/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [02:42<00:00, 289kB/s] 


Training started...
Epoch 1 loss: 0.357
Epoch 2 loss: 0.081
Epoch 3 loss: 0.047

--- Results ---
                       precision    recall  f1-score   support

                  ALL       1.00      1.00      1.00        33
         Brain Cancer       1.00      1.00      1.00        32
        Breast Cancer       1.00      1.00      1.00        29
      Cervical Cancer       0.94      1.00      0.97        29
        Kidney Cancer       1.00      1.00      1.00        26
Lung and Colon Cancer       1.00      0.95      0.97        38
             Lymphoma       1.00      1.00      1.00        22
          Oral Cancer       1.00      1.00      1.00        31

             accuracy                           0.99       240
            macro avg       0.99      0.99      0.99       240
         weighted avg       0.99      0.99      0.99       240

DONE - Model saved!


In [2]:
%pip install torch torchvision scikit-learn pillow

  Using cached torch-2.14.1-cp311-cp311-win_amd64.whl.metadata (38 kB)
  Using cached torchvision-0.29.1-cp311-cp311-win_amd64.whl.metadata (5.7 kB)
  Using cached filelock-4.0.8-py3-none-any.whl.metadata (2.0 kB)
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached torch-2.14.1-cp311-cp311-win_amd64.whl (124.1 MB)
Using cached torchvision-0.29.1-cp311-cp311-win_amd64.whl (1.4 MB)
Using cached fsspec-2026.9.0-py3-none-any.whl (221 kB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Using cached setuptools-84.0.0-py3-none-any.whl (818 kB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached filelock-4.0.8-py3-none-any.whl (109 kB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
  Attempt


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os, json, random, torch, torch.nn as nn, pandas as pd
from torchvision import models, transforms
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
BASE = r"C:\Users\ok\prism-dr"
print("torch", torch.__version__)

torch 2.14.1+cpu


In [4]:
f, y = te_i[0]
with torch.no_grad():
    p = torch.softmax(m(T_te(Image.open(f).convert("RGB")).unsqueeze(0)), 1)[0]
print("true:", classes[y], "| predicted:", classes[p.argmax()], f"| confidence {p.max().item()*100:.1f}%")

true: Breast Cancer | predicted: Breast Cancer | confidence 99.4%


In [5]:
import os, json
assert os.path.exists(BASE + r"\models\m2_resnet18.pt"), "Missing model file!"
assert os.path.exists(BASE + r"\models\m2_classes.json"), "Missing classes file!"
cl = json.load(open(BASE + r"\models\m2_classes.json"))
print("All files present. Classes count:", len(cl))

All files present. Classes count: 8
